# Python 클래스(Class) 필수 정리

Python에서 클래스 관련 **반드시 알아야 하는 핵심 사항**을 예제와 함께 정리한 노트북입니다.

## 목차
1. 클래스 기본 문법과 `__init__`
2. 인스턴스 속성 vs 클래스 속성 (mutable 클래스 속성 함정!)
3. 메서드 3종: 인스턴스 / 클래스 / 정적 메서드
4. 상속과 `super()`, 메서드 오버라이딩
5. MRO (Method Resolution Order)와 다중 상속
6. 던더(dunder) 매직 메서드
7. `@property` — 게터/세터 대신
8. 캡슐화 관례: `_protected`와 `__private`(name mangling)
9. 데이터클래스 `@dataclass`와 `__slots__`
10. 추상 클래스 (ABC)
11. 자주 하는 실제 실수 모음


## 1. 클래스 기본 문법과 `__init__`

- `class` 키워드로 정의하며, 클래스 이름은 관례상 **PascalCase**를 사용합니다(Python 개선 제안 PEP 8).
- `__init__`은 **생성자가 아니라 초기화 메서드**입니다. 객체는 이미 만들어진 후 `__init__`이 호출됩니다.
- 첫 번째 매개변수 `self`는 **호출된 인스턴스 자신**을 가리키며, 이름은 관례일 뿐이지만 반드시 지켜야 합니다.
- 속성은 반드시 `__init__` 안에서 `self.속성 = 값`으로 초기화하는 것이 좋습니다.

In [ ]:
class Dog:
    # 클래스 본문에 직접 정의한 함수 = 메서드
    def __init__(self, name: str, age: int):
        # __init__: 인스턴스 생성 직후 호출되는 초기화 메서드
        self.name = name   # 인스턴스 속성
        self.age = age

    def bark(self) -> str:  # 인스턴스 메서드 — self 필수
        return f"{self.name}가 짖는다: 멍멍!"

# 인스턴스 생성 — __init__의 self에는 자동으로 객체가 전달됨
dog = Dog("바둑이", 3)
print(dog.name)
print(dog.bark())          # dog.bark() == Dog.bark(dog)와 동일
print(Dog.bark(dog))       # 내부적으로는 이렇게 동작

## 2. 인스턴스 속성 vs 클래스 속성 ⭐ 가장 흔한 함정

- **클래스 속성**: 클래스에 정의 — 모든 인스턴스가 **공유**
- **인스턴스 속성**: `self.xxx = ...`로 할당 — 인스턴스 각각의 값
- 조회 순서: **인스턴스 → 클래스 → 부모 클래스** 순으로 탐색
- ⚠️ **함정**: 클래스 속성에 **가변 객체(list, dict 등)**를 두면 모든 인스턴스가 같은 객체를 공유합니다!

In [ ]:
class Counter:
    count = 0            # 클래스 속성 (모든 인스턴스 공유)
    items = []           # ⚠️ 위험: 모든 인스턴스가 같은 리스트를 공유!

    def __init__(self):
        Counter.count += 1   # 클래스 속성 수정은 클래스 이름으로

a, b = Counter(), Counter()
print(a.count, b.count)   # 2 2 — 공유됨

a.items.append("x")       # ⚠️ b.items에도 "x"가 보임
print(b.items)

In [ ]:
# 올바른 패턴: 가변 상태는 __init__에서 인스턴스 속성으로
class CounterSafe:
    total = 0                        # 진짜 '공유'하고 싶은 것만 클래스 속성으로

    def __init__(self):
        self.items: list[str] = []   # 인스턴스마다 별도의 리스트

    def add(self, item: str) -> None:
        self.items.append(item)      # 인스턴스 속성만 건드림
        CounterSafe.total += 1

c1, c2 = CounterSafe(), CounterSafe()
c1.add("a"); c2.add("b")
print(c1.items, c2.items)  # 서로 독립
print(CounterSafe.total)   # 공유 카운터만 2

## 3. 메서드 3종: 인스턴스 / 클래스 / 정적

| 종류 | 데코레이터 | 첫 매개변수 | 용도 |
|---|---|---|---|
| 인스턴스 메서드 | 없음 | `self` | 인스턴스 상태 사용/수정 |
| 클래스 메서드 | `@classmethod` | `cls` | 대체 생성자, 클래스 상태 접근 |
| 정적 메서드 | `@staticmethod` | 없음 | 클래스 내부에 두는 단순 유틸 함수 |

**대체 생성자(alternative constructor)** 패턴이 `@classmethod`의 대표 용도입니다.

In [ ]:
class Temperature:
    def __init__(self, celsius: float):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f: float) -> "Temperature":
        # 대체 생성자: cls로 같은 클래스의 인스턴스를 만듦 (상속에서도 올바르게 동작)
        return cls((f - 32) * 5 / 9)

    @staticmethod
    def is_freezing(celsius: float) -> bool:
        # self/cls가 필요 없는 순수 유틸 함수
        return celsius <= 0

t = Temperature.from_fahrenheit(32)   # 0°C
print(t.celsius, Temperature.is_freezing(t.celsius))

## 4. 상속과 `super()`

- `(Parent)`를 괄호에 쓰면 상속. 메서드를 재정의하면 **오버라이딩**됩니다.
- `super()`는 부모 클래스의 구현을 호출 — 특히 오버라이딩한 `__init__`에서 부모 초기화를 잊지 말 것.
- `isinstance()`로 타입 확인, `issubclass()`로 상속 관계 확인.

In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name

    def speak(self) -> str:
        return "..."

class Cat(Animal):
    def __init__(self, name: str, indoor: bool):
        super().__init__(name)     # ⭐ 부모 초기화 필수 — 잊으면 name이 없음
        self.indoor = indoor

    def speak(self) -> str:        # 오버라이딩
        return f"{self.name}: 야옹"

    def speak_loud(self) -> str:
        return super().speak().upper() + "!! 야옹!!"  # 부모 구현 재사용

cat = Cat("나비", True)
print(cat.speak(), "|", cat.speak_loud())
print(isinstance(cat, Animal), issubclass(Cat, Animal))

## 5. MRO와 다중 상속

- Python은 다중 상속을 지원하며, 메서드 탐색 순서는 **MRO**(Method Resolution Order)로 결정됩니다.
- MRO는 C3 선형화 알고리즘으로 계산되며 `ClassName.__mro__` 또는 `mro()`로 확인할 수 있습니다.
- 다이아몬드 상속에서도 각 부모가 정확히 한 번만 호출되도록 `super()`가 다음 MRO 항목을 찾아줍니다.

In [ ]:
class A:
    def hello(self): return "A"

class B(A):
    def hello(self):
        print("B ->", super().hello()); return "B"

class C(A):
    def hello(self):
        print("C ->", super().hello()); return "C"

class D(B, C):   # 다이아몬드: D -> B -> C -> A
    pass

print([cls.__name__ for cls in D.__mro__])
D().hello()   # A의 hello는 한 번만 호출됨

## 6. 던더(dunder) 매직 메서드

특수 메서드를 구현하면 Python 연산자·내장 함수와 객체를 연결할 수 있습니다.

| 메서드 | 역할 |
|---|---|
| `__repr__` | 개발자용 표현 (디버깅, REPL) — **거의 항상 구현 권장** |
| `__str__` | 사용자용 표현 (`print`) — 없으면 `__repr__` 사용 |
| `__eq__` | `==` 비교 (구현하면 해시 가능성에 주의) |
| `__len__` | `len()` |
| `__getitem__` | `obj[key]` 인덱싱, for문 순회까지 지원됨 |
| `__iter__` / `__next__` | 반복자 프로토콜 |

In [ ]:
class Book:
    def __init__(self, title: str, pages: int):
        self.title = title
        self.pages = pages

    def __repr__(self) -> str:          # 디버깅용 표현
        return f"Book(title={self.title!r}, pages={self.pages})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Book):
            return NotImplemented       # 다른 타입과의 비교는 파이썬에 위임
        return (self.title, self.pages) == (other.title, other.pages)

    def __len__(self) -> int:
        return self.pages

    def __getitem__(self, idx: int) -> str:
        return self.title[idx]          # __getitem__만으로 for 순회 가능!

book = Book("파이썬", 300)
print(book)                             # __repr__ 사용
print(book == Book("파이썬", 300), len(book))
print([ch for ch in book][:5])          # for ... in book 동작

## 7. `@property` — 게터/세터 대신

- Java 스타일의 `getX()/setX()` 대신 Python은 `@property`로 **속성처럼 보이면서 로직을 끼워 넣습니다**.
- 읽기 전용 속성, 유효성 검사, 계산된 속성에 사용.
- 처음엔 일반 속성으로 만들고, 나중에 로직이 필요해질 때 `@property`로 바꿔도 **호출부 코드가 안 바뀐다는 점**이 핵심 장점.

In [ ]:
class Circle:
    def __init__(self, radius: float):
        self.radius = radius            # setter를 통해 검증됨!

    @property
    def radius(self) -> float:
        return self._radius

    @radius.setter
    def radius(self, value: float) -> None:
        if value < 0:
            raise ValueError("반지름은 음수일 수 없습니다")
        self._radius = value

    @property
    def area(self) -> float:            # 읽기 전용, 계산된 속성
        return 3.141592 * self._radius ** 2

c = Circle(5)
print(c.area)          # 메서드 호출 없이 속성처럼 접근
c.radius = 10          # setter 실행 (검증 포함)
print(c.area)
try:
    c.radius = -1      # 검증 실패 → 예외
except ValueError as e:
    print("오류:", e)

## 8. 캡슐화 관례: `_protected`와 `__private`

Python에는 진짜 접근 제한자가 없고 **관례(naming convention)**로 구분합니다.

- `_name`: "내부용" 관례 표시. 접근은 가능하지만 외부에서 쓰지 말 것.
- `__name`: **name mangling** — `_ClassName__name`으로 자동 변경되어 상속 클래스와의 충돌을 방지. 강한 은닉이 아니라 **충돌 방지 목적**.

In [ ]:
class Account:
    def __init__(self, balance: int):
        self._balance = balance      # 관례상 내부 속성
        self.__pin = "1234"          # name mangling 대상

    def get_balance(self) -> int:
        return self._balance

acc = Account(10000)
print(acc._balance)          # 접근 가능하지만 '하지 말라'는 신호
try:
    print(acc.__pin)         # AttributeError!
except AttributeError:
    print("acc.__pin 접근 불가")
print(acc._Account__pin)     # mangling된 실제 이름 — 우회는 가능(비추천)

## 9. 데이터클래스 `@dataclass`와 `__slots__`

- `@dataclass`는 `__init__`, `__repr__`, `__eq__` 등을 **자동 생성** — 단순 데이터 담는 클래스는 무조건 이걸 우선 고려.
- `frozen=True`로 불변 객체를 만들 수 있고, `field(default_factory=...)`로 가변 기본값 함정을 피합니다.
- `__slots__`는 속성 목록을 고정해 **메모리를 아끼고** 오타 속성 생성을 방지합니다.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float = 0.0
    tags: list[str] = field(default_factory=list)   # ⚠️ [] 직접 쓰면 모든 인스턴스가 공유!

    @property
    def dist(self) -> float:
        return (self.x ** 2 + self.y ** 2) ** 0.5

p1, p2 = Point(3, 4), Point(1)
print(p1)                     # 자동 생성된 __repr__
print(p1 == Point(3, 4))      # 자동 생성된 __eq__
print(p1.dist)

@dataclass(frozen=True)
class Config:
    host: str
    port: int = 8080

cfg = Config("localhost")
try:
    cfg.port = 9090           # frozen → 수정 불가
except Exception as e:
    print(type(e).__name__, "-", e)

In [ ]:
class Compact:
    __slots__ = ("a", "b")      # 허용된 속성 고정

    def __init__(self):
        self.a = 1
        self.b = 2

obj = Compact()
print(obj.a, obj.b)
try:
    obj.c = 3                   # 슬롯 밖 속성 → AttributeError
except AttributeError:
    print("obj.c 생성 불가 — 오타 방지 + 메모리 절약")

## 10. 추상 클래스 (ABC)

- **반드시 구현해야 하는 메서드를 강제**하려면 `abc.ABC`와 `@abstractmethod`를 사용합니다.
- 추상 메서드가 하나라도 구현 안 된 클래스는 **인스턴스화 자체가 불가능**합니다.
- 인터페이스 역할이 필요할 때(여러 구현체가 같은 계약을 지켜야 할 때) 사용.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

    def describe(self) -> str:      # 추상 클래스도 일반 메서드 가능
        return f"넓이: {self.area():.2f}"

class Rect(Shape):
    def __init__(self, w: float, h: float):
        self.w, self.h = w, h
    def area(self) -> float:
        return self.w * self.h

r = Rect(3, 4)
print(r.describe())

try:
    Shape()                          # 추상 클래스는 인스턴스화 불가
except TypeError as e:
    print("오류:", e)

## 11. 자주 하는 실수 모음

### 실수 1: 가변 기본값 / 가변 클래스 속성 (위 2절 참고)
### 실수 2: `__init__` 오버라이드 시 부모 초기화 누락
### 실수 3: `self`를 잊은 메서드 호출, `cls`/`self` 혼동
### 실수 4: `__eq__`만 구현하고 `__hash__`를 깨뜨림

In [ ]:
class Item:
    def __init__(self, name: str):
        self.name = name

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Item):
            return NotImplemented
        return self.name == other.name
    # __eq__를 정의하면 기본 __hash__가 사라져 dict/set에서 사용 불가!

try:
    {Item("a"): 1}
except TypeError as e:
    print("오류:", e)

# 해결: __hash__도 함께 정의
class ItemOK:
    def __init__(self, name: str):
        self.name = name
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, ItemOK):
            return NotImplemented
        return self.name == other.name
    def __hash__(self) -> int:
        return hash(self.name)      # 불변 필드 기반 해시

print({ItemOK("a"): 1})

## 핵심 요약

1. **`__init__`은 초기화 메서드**, `self`는 호출된 인스턴스 자신.
2. **가변 클래스 속성/기본값 금지** — 공유 상태 버그의 최대 원인.
3. `@classmethod`는 **대체 생성자**, `@staticmethod`는 단순 유틸.
4. 상속 시 `super().__init__()` 잊지 말기; 다중 상속 동작은 **MRO**로 확인.
5. `__repr__`, `__eq__`(필요하면 `__hash__`) 등 **던더 메서드**로 Python스럽게.
6. 게터/세터 대신 **`@property`**.
7. 단순 데이터는 **`@dataclass`** 우선, 성능·오타 방지는 **`__slots__`**.
8. 구현 강제는 **추상 클래스(ABC)**.